In [2]:
# First we load the packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf

# I. Data Preparation

In [19]:
# Read in master data
# Note this master dataset was created in the Week 2 assignment
Master = pd.read_csv("../Data/Master.csv")

In [20]:
Master.columns

Index(['Unnamed: 0', 'playerID', 'yearID', 'stint', 'G', 'AB', 'R', 'H',
       'Doubles', 'Triples', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP',
       'SH', 'SF', 'GIDP', 'PA', 'OBP', 'SLG', 'SalYear', 'teamID', 'lgID',
       'salary', 'lnSal', 'debutyr', 'Exp', 'Arb', 'Free', 'POS', 'Catch',
       'Infld'],
      dtype='object')

In [21]:
#Your Code Here
Master['Exp2'] = Master['Exp']**2
Master['BA'] = Master['H']/Master['AB']
Master['IP'] = Master['SLG'] - Master['BA']
Master['Eye'] = (Master['HBP']+Master['BB'])/Master['PA']
Master = Master[(Master.SalYear >= 1995) & (Master.SalYear <= 2015)]

In [22]:
MB_Seas.groupby(['yearID'])['BA'].median().sort_values()

yearID
2013    0.258850
2014    0.259067
2012    0.260116
2011    0.262000
2010    0.262735
2002    0.265221
2009    0.266667
2001    0.267102
1995    0.268750
2003    0.268786
1997    0.269674
2005    0.271157
1998    0.271215
2008    0.271357
2004    0.272627
2007    0.272727
1994    0.273063
2000    0.274038
1996    0.274958
2006    0.276968
1999    0.277868
Name: BA, dtype: float64

# II. Running Regressions for Each Season

In [34]:
# Write a function to run the Moneyball regression annually for free agents only
def MBExpandFA(Season):
    MB_Seas = Master[(Master.SalYear == Season) & (Master.Free == 1)]
    global lm
    lm = smf.ols(formula = 'lnSal ~ BA + IP + Eye + PA + Exp + Exp2 + C(POS)', data=MB_Seas).fit()
    return;

In [35]:
#Your Code Here
index = 0
lm_Results = [0]
for index in range(1,22):
    lm_Results.append(index)
    index = index + 1
display(lm_Results) 

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

In [36]:
Season = 1995
i = 0
while Season <= 2015:
    MBExpandFA(Season)
    lm_Results[i] = lm
    i = i + 1
    Season = Season + 1

In [37]:
Season = 1995
lm_Season = ["1995"]
for Season in range(1996, 2016):
    lm_Season.append(str(Season))
    Season = Season + 1

In [38]:
#Create a list of season names to label regression results and divide list into eras
    
Pre_MB = lm_Season[:6]
MB_Period = lm_Season[6:14]
Post_MB = lm_Season[14:20]

print(Pre_MB)
print(MB_Period)
print(Post_MB)

['1995', '1996', '1997', '1998', '1999', '2000']
['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008']
['2009', '2010', '2011', '2012', '2013', '2014']


In [39]:
info_dict={'R-squared' : lambda x: f"{x.rsquared:.2f}",
           'No. observations' : lambda x: f"{int(x.nobs):d}"}

In [40]:
# Regression results from 1995-2000
from statsmodels.iolib.summary2 import summary_col
PreMB_Out = summary_col([lm_Results[0],lm_Results[1],lm_Results[2],lm_Results[3],lm_Results[4],
                        lm_Results[5]], model_names=Pre_MB, 
                        stars=True, info_dict = info_dict)
print(PreMB_Out)


                    1995      1996       1997       1998       1999       2000   
---------------------------------------------------------------------------------
Intercept        9.9694*** 11.3986*** 12.2387*** 10.1032*** 10.3032*** 11.7979***
                 (1.1036)  (1.3914)   (1.0950)   (1.0611)   (0.9596)   (0.8532)  
C(POS)[T.2B]     -0.8179** -0.1286    -0.4298    -0.0151    -0.2160    -0.2726   
                 (0.3753)  (0.3466)   (0.2642)   (0.2872)   (0.2294)   (0.2175)  
C(POS)[T.3B]     -0.2760   -0.3602    -0.5173*   -0.0798    0.2404     -0.1103   
                 (0.3050)  (0.3140)   (0.2694)   (0.2479)   (0.2184)   (0.2205)  
C(POS)[T.C]      -0.0799   0.0075     -0.2965    0.1119     -0.0514    -0.0740   
                 (0.3222)  (0.3241)   (0.2630)   (0.2549)   (0.2281)   (0.2163)  
C(POS)[T.DH]     -0.5695   0.3089     -0.4400    -0.2474    0.1792     -0.3156   
                 (0.4073)  (0.4124)   (0.2920)   (0.2946)   (0.2432)   (0.2642)  
C(POS)[T.OF]   

In [41]:
# Regression results from 2001-2008
MB_Out = summary_col([lm_Results[6],lm_Results[7],lm_Results[8],lm_Results[9],lm_Results[10],
                        lm_Results[11],lm_Results[12],lm_Results[13]], model_names=MB_Period, 
                        stars=True,info_dict = info_dict)
print(MB_Out)


                    2001       2002       2003       2004       2005       2006       2007       2008   
--------------------------------------------------------------------------------------------------------
Intercept        11.2572*** 10.3452*** 10.0969*** 10.7180*** 10.4152*** 10.6209*** 10.8980*** 11.6518***
                 (0.7926)   (1.0372)   (1.1665)   (1.1329)   (0.9773)   (0.9818)   (0.7737)   (0.9852)  
C(POS)[T.2B]     0.1785     -0.0029    -0.2719    0.0510     -0.4057    -0.1014    -0.2304    0.0278    
                 (0.2204)   (0.2865)   (0.2921)   (0.2977)   (0.2602)   (0.2517)   (0.2269)   (0.2562)  
C(POS)[T.3B]     0.2367     0.2321     0.0073     -0.0236    -0.0976    0.3954*    0.1752     0.5582**  
                 (0.2175)   (0.3068)   (0.2818)   (0.2966)   (0.2743)   (0.2369)   (0.2308)   (0.2338)  
C(POS)[T.C]      0.2604     0.3305     0.4778*    0.1804     -0.0563    0.2727     0.3856*    0.3537    
                 (0.2244)   (0.2679)   (0.2740)   (0.2

In [42]:
# Regression results from 2009-2014
PostMB_Out = summary_col([lm_Results[14],lm_Results[15],lm_Results[16],lm_Results[17],lm_Results[18],
                        lm_Results[19]], model_names=Post_MB, 
                        stars=True,info_dict = info_dict)
print(PostMB_Out)


                    2009       2010      2011      2012       2013       2014   
--------------------------------------------------------------------------------
Intercept        10.7617*** 8.9933*** 9.8735*** 12.8252*** 12.8365*** 10.5028***
                 (0.9933)   (1.2090)  (1.2633)  (1.0809)   (1.2736)   (1.1818)  
C(POS)[T.2B]     -0.1089    0.0035    0.2046    0.1738     -0.5499**  0.3901    
                 (0.3495)   (0.3420)  (0.3225)  (0.3226)   (0.2748)   (0.2568)  
C(POS)[T.3B]     0.3158     0.4651    -0.0762   0.4312     -0.2036    0.2050    
                 (0.2555)   (0.3045)  (0.3072)  (0.3012)   (0.2614)   (0.2823)  
C(POS)[T.C]      0.6320**   0.1701    0.0726    0.2070     -0.2055    0.2757    
                 (0.2730)   (0.3241)  (0.3470)  (0.3352)   (0.2767)   (0.2650)  
C(POS)[T.DH]     0.2828     0.1143    -0.5956*  0.4847     -0.3438    0.3179    
                 (0.3487)   (0.3786)  (0.3278)  (0.3884)   (0.3285)   (0.3415)  
C(POS)[T.OF]     0.3645*   

# III. Running the Pooled Regression

In [43]:
#Your Code Here
Master_Free = Master[Master.Free == 1].copy()
Master_Free['PreMB'] = np.where(Master_Free['SalYear']<2004,1,0)
Master_Free.describe()

,Unnamed: 0,yearID,stint,G,AB,R,H,Doubles,Triples,HR,...,Exp,Arb,Free,Catch,Infld,Exp2,BA,IP,Eye,PreMB
count,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,...,2841.000000,2841.0,2841.0,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000,2841.000000
mean,3942.336501,2003.809574,1.236536,119.377332,408.487504,59.135164,112.635340,22.287575,1.947906,14.468145,...,10.069342,0.0,1.0,0.137980,0.322422,109.247800,0.271475,0.163928,0.101523,0.437170
std,2269.965795,5.835056,0.712822,30.719136,145.259956,28.484541,45.661153,10.326979,2.182863,11.390127,...,2.803377,0.0,0.0,0.344939,0.467486,66.079427,0.032331,0.065851,0.037816,0.496124
min,8.000000,1994.000000,1.000000,34.000000,130.000000,6.000000,21.000000,1.000000,0.000000,0.000000,...,7.000000,0.0,1.0,0.000000,0.000000,49.000000,0.147929,0.019608,0.013514,0.000000
25%,1983.000000,1999.000000,1.000000,97.000000,283.000000,35.000000,74.000000,14.000000,0.000000,6.000000,...,8.000000,0.0,1.0,0.000000,0.000000,64.000000,0.250591,0.115632,0.075581,0.000000
50%,3950.000000,2004.000000,1.000000,125.000000,428.000000,58.000000,115.000000,21.000000,1.000000,12.000000,...,9.000000,0.0,1.0,0.000000,0.000000,81.000000,0.272040,0.157895,0.096591,0.000000
75%,5901.000000,2009.000000,1.000000,146.000000,535.000000,80.000000,150.000000,30.000000,3.000000,21.000000,...,12.000000,0.0,1.0,0.000000,1.000000,144.000000,0.293173,0.203774,0.124172,1.000000
max,7780.000000,2014.000000,10.000000,163.000000,716.000000,152.000000,227.000000,57.000000,21.000000,73.000000,...,24.000000,0.0,1.0,1.000000,1.000000,576.000000,0.393795,0.535714,0.390600,1.000000


In [45]:
Pooled_lm = smf.ols(formula = 'lnSal ~ BA + IP + Eye + PA + Exp + Exp2 + C(POS) + PreMB*(BA + IP + Eye + PA + Exp + Exp2 + C(POS))',\
                    data=Master_Free).fit()
Pooled_lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  lnSal   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.516
Method:                 Least Squares   F-statistic:                     122.2
Date:                Fri, 03 Apr 2026   Prob (F-statistic):               0.00
Time:                        11:47:09   Log-Likelihood:                -3313.7
No. Observations:                2841   AIC:                             6679.
Df Residuals:                    2815   BIC:                             6834.
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             11.5566      0.296     39.071      0.000      10.977      12.137
C(POS)[T.2B]          -0.1073      0.083     -1.290      0.197      -0.270       0.056
C(POS)[T.3B]           0.1828      0.079      2.312      0.021       0.028       0.338
C(POS)[T.C]            0.0918      0.079      1.164      0.245      -0.063       0.246
C(POS)[T.DH]           0.0485      0.098      0.497      0.619      -0.143       0.240
C(POS)[T.OF]           0.1269      0.064      1.982      0.048       0.001       0.252
C(POS)[T.SS]           0.1239      0.086      1.435      0.152      -0.045       0.293
BA                     2.9810      0.687      4.338      0.000       1.634       4.328
IP                     2.1509      0.395      5.445      0.000       1.376       2.925
Eye                    3.7615      0.640      5.874      0.000       2.506       5.017
PA                     0.0035      0.000     26.384      0.000       0.003       0.004
Exp                    0.0961      0.041      2.353      0.019       0.016       0.176
Exp2                  -0.0044      0.002     -2.560      0.011      -0.008      -0.001
PreMB                 -0.6327      0.463     -1.367      0.172      -1.540       0.275
PreMB:C(POS)[T.2B]    -0.0952      0.128     -0.746      0.456      -0.346       0.155
PreMB:C(POS)[T.3B]    -0.2705      0.121     -2.232      0.026      -0.508      -0.033
PreMB:C(POS)[T.C]     -0.0250      0.121     -0.206      0.837      -0.263       0.213
PreMB:C(POS)[T.DH]    -0.1985      0.149     -1.332      0.183      -0.491       0.094
PreMB:C(POS)[T.OF]    -0.1558      0.097     -1.602      0.109      -0.346       0.035
PreMB:C(POS)[T.SS]    -0.0129      0.132     -0.098      0.922      -0.273       0.247
PreMB:BA              -0.9981      1.035     -0.964      0.335      -3.028       1.032
PreMB:IP               1.3514      0.555      2.436      0.015       0.264       2.439
PreMB:Eye             -2.4039      0.894     -2.690      0.007      -4.156      -0.652
PreMB:PA            2.091e-05      0.000      0.101      0.919      -0.000       0.000
PreMB:Exp              0.0722      0.065      1.109      0.268      -0.055       0.200
PreMB:Exp2            -0.0032      0.003     -1.169      0.242      -0.009       0.002
==============================================================================
Omnibus:                       15.145   Durbin-Watson:                   1.331
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               20.745
Skew:                           0.027   Prob(JB):                     3.13e-05
Kurtosis:                       3.415   Cond. No.                     4.72e+04
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is corr